# PEFT Methods — Building Blocks and Visualisations

Hands-on anatomy guide for six parameter-efficient adaptation strategies applied to a
pretrained **ViT-Small** (`WinKawaks/vit-small-patch16-224`, 22 M parameters, hidden dim 384).

Each section: (1) states the idea, (2) shows which parameters train, (3) visualises the mechanics.

| # | Method | What trains | Where it acts |
|---|---|---|---|
| 1 | Linear probe | classifier head | output / label space |
| 2 | BitFit | bias vectors only | small existing param subset |
| 3 | Visual prompt tuning | learnable pixel patch | input image space |
| 4 | LoRA | A and B matrices | weight space (low-rank delta) |
| 5 | Adapter | bottleneck residual | hidden-state path |
| 6 | Partial fine-tuning | last transformer block | checkpoint weights |

In [ ]:
# !pip install -q torch torchvision transformers peft matplotlib pandas
import sys, copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from transformers import ViTModel

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    sys.path.insert(0, str((ROOT / "..").resolve()))
elif (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT.resolve()))

from src.methods.linear_probe import LinearProbeModel
from src.methods.adapters import AdapterHeadClassifier
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.lora import LoRAClassifier
from src.methods.bitfit import BitFitClassifier
from src.methods.partial_ft import PartialFineTuneClassifier
from src.training import count_trainable_parameters, freeze_module

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "WinKawaks/vit-small-patch16-224"
N_CLASSES  = 10

class ViTBackbone(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.feature_dim = self.vit.config.hidden_size

    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]  # CLS token

backbone = ViTBackbone().to(device)
freeze_module(backbone)

cfg   = backbone.vit.config
total = sum(p.numel() for p in backbone.parameters())
print(f"Model        : {MODEL_NAME}")
print(f"Hidden dim   : {cfg.hidden_size}")
print(f"Num layers   : {cfg.num_hidden_layers}")
print(f"Num heads    : {cfg.num_attention_heads}")
print(f"Total params : {total:,}")
D = backbone.feature_dim

In [ ]:
# Parameter breakdown by top-level component
rows = [{"component": name.split(".")[0], "params": p.numel()}
        for name, p in backbone.vit.named_parameters()]
summary = pd.DataFrame(rows).groupby("component")["params"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3))
summary.plot.bar(ax=ax, color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_title("ViT-Small: parameter count by top-level component", fontsize=12)
ax.set_ylabel("Parameters")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()
print(summary.to_string())

---
## 1. Linear Probe

**Idea:** The pretrained backbone is frozen. Only a single linear layer (the classifier head)
is trained on the new task.

```
input → [frozen ViT] → CLS feature h ∈ ℝᴰ → [W, b  trainable] → logits
```

**When to use:** First baseline when you believe pretrained features are already
linearly separable for your task.

In [ ]:
lp = LinearProbeModel(copy.deepcopy(backbone), D, N_CLASSES)

total_p = sum(p.numel() for p in lp.parameters())
train_p = count_trainable_parameters(lp)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.3f}%)")
print()
for name, p in lp.named_parameters():
    if p.requires_grad:
        print(f"  {name:35s} shape={list(p.shape)}  numel={p.numel()}")

W = lp.head.weight.detach().cpu()      # [N_CLASSES, D]
fig, ax = plt.subplots(figsize=(9, 2))
v = W.abs().max().item()
im = ax.imshow(W.numpy(), cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.015)
ax.set_xlabel(f"Feature dim  (D={W.shape[1]})")
ax.set_ylabel(f"Classes ({W.shape[0]})")
ax.set_title(f"Linear probe head  W ∈ ℝ^{{{W.shape[0]}×{W.shape[1]}}}  — the only trainable matrix")
plt.tight_layout()
plt.show()

---
## 2. BitFit

**Idea:** Freeze all weight *matrices*; keep every *bias vector* and LayerNorm β trainable.

```
y = W x + b    →    W frozen,   b trainable
```

**Why it can work:** Biases shift the activation distribution without rotating the
learned feature directions — a lightweight way to adapt to a domain shift.

**When to use:** Very tight parameter budgets, or as a cheap ablation to see whether
direction or offset is the bottleneck.

In [ ]:
bf = BitFitClassifier(copy.deepcopy(backbone), D, N_CLASSES)

total_p = sum(p.numel() for p in bf.parameters())
train_p = count_trainable_parameters(bf)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.3f}%)")

rows = [{"name": n, "numel": p.numel(),
         "trainable": p.requires_grad,
         "kind": "bias / head" if ("bias" in n or n.startswith("head")) else "weight"}
        for n, p in bf.named_parameters()]
df_bf = pd.DataFrame(rows)
print()
print(df_bf.groupby(["kind", "trainable"])["numel"].sum().to_string())

# Bar chart: each param group, red=trainable, grey=frozen
fig, ax = plt.subplots(figsize=(11, 2.8))
colours = ["#e06c75" if r.trainable else "#abb2bf" for _, r in df_bf.iterrows()]
ax.bar(range(len(df_bf)), df_bf["numel"], color=colours, width=1.0)
ax.set_yscale("log")
ax.set_xlabel("Parameter group index (each bar = one named parameter tensor)")
ax.set_ylabel("Parameters (log scale)")
ax.set_title("BitFit: trainable (red) vs frozen (grey) parameter groups")
ax.legend(handles=[mpatches.Patch(color="#e06c75", label="trainable  (bias / head)"),
                   mpatches.Patch(color="#abb2bf", label="frozen  (weight matrices)")])
plt.tight_layout()
plt.show()

---
## 3. Visual Prompt Tuning (VPT)

**Idea:** Add a small learnable perturbation to the input image before feeding it
into the frozen backbone.

```
x_prompted = x + prompt(clipped to image corners)  →  [frozen ViT]  →  head
```

Our implementation adds the prompt to the **top-left corner** (image-space patch).
The original VPT paper (Jia et al. 2022) prepends learnable tokens to the *token sequence*
inside the ViT — both approaches steer the frozen model by changing what it sees.

**What trains:** A small tensor (C × P × P) plus the head.

In [ ]:
vpt = PromptTunedClassifier(copy.deepcopy(backbone), D, N_CLASSES, prompt_size=32)

total_p = sum(p.numel() for p in vpt.parameters())
train_p = count_trainable_parameters(vpt)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.4f}%)")
print(f"Prompt tensor : {list(vpt.prompt.prompt.shape)}")

torch.manual_seed(7)
img = (torch.rand(3, 224, 224) * 0.5 + 0.25).clamp(0, 1)
prompt_val = vpt.prompt.prompt.detach().cpu()[0]   # [3, 32, 32]
ph, pw = prompt_val.shape[-2:]
img_p = img.clone()
img_p[:, :ph, :pw] = (img_p[:, :ph, :pw] + prompt_val).clamp(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))

axes[0].imshow(img.permute(1, 2, 0).numpy())
axes[0].set_title("Original image"); axes[0].axis("off")

axes[1].imshow(img_p.permute(1, 2, 0).numpy())
axes[1].add_patch(plt.Rectangle((0, 0), pw, ph, fill=False, edgecolor="red", linewidth=2))
axes[1].set_title("Prompted image\n(red = prompt area)"); axes[1].axis("off")

v = prompt_val.abs().max().item()
im2 = axes[2].imshow(prompt_val.mean(0).numpy(), cmap="RdBu", vmin=-v, vmax=v)
plt.colorbar(im2, ax=axes[2], fraction=0.06)
axes[2].set_title(f"Prompt values\n(mean over 3 channels, {ph}×{pw} patch)")

diff = (img_p - img).abs().sum(0)
im3 = axes[3].imshow(diff.numpy(), cmap="hot")
plt.colorbar(im3, ax=axes[3], fraction=0.06)
axes[3].set_title("Absolute pixel change\n(non-zero only in patch area)")

plt.suptitle("VPT: a learnable perturbation steers the frozen ViT by changing its input", y=1.02)
plt.tight_layout()
plt.show()

---
## 4. LoRA — Low-Rank Adaptation

**Idea:** Decompose the weight update as a product of two small matrices:

```
W_eff = W₀ + ΔW = W₀ + B · A
```

- A ∈ ℝ^{r×d_in},  B ∈ ℝ^{d_out×r},  rank r ≪ d
- A initialised from Gaussian;  B initialised to **zero** → ΔW = 0 at the start
- W₀ is never updated; only A and B accumulate gradients

**Parameter saving:**  r·(d_in + d_out)  vs  d_in·d_out  for a full update.

Applied via the **PEFT library** to the `query` and `value` projections in every
attention layer.

In [ ]:
RANK = 8
lora_m = LoRAClassifier(copy.deepcopy(backbone), D, N_CLASSES,
                        target_modules=["query", "value"], rank=RANK)

total_p = sum(p.numel() for p in lora_m.parameters())
train_p = count_trainable_parameters(lora_m)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.2f}%)")

def find_first_lora(model):
    for name, module in model.named_modules():
        if hasattr(module, "lora_A") and module.lora_A:
            key = next(iter(module.lora_A))
            A  = module.lora_A[key].weight.detach().cpu()
            B  = module.lora_B[key].weight.detach().cpu()
            W0 = module.weight.detach().cpu()
            return name, W0, A, B
    raise RuntimeError("No LoRA layer found")

name, W0, A, B = find_first_lora(lora_m)
dW = B @ A
print(f"\nFirst LoRA layer : {name}")
print(f"W₀ shape : {tuple(W0.shape)}   ({W0.numel():,} params — frozen)")
print(f"A  shape : {tuple(A.shape)}")
print(f"B  shape : {tuple(B.shape)}   combined {A.numel()+B.numel():,} trainable params")
print(f"Rank of ΔW=BA : {int(torch.linalg.matrix_rank(dW))}  (equals r={RANK} by construction)")
print(f"Compression   : {W0.numel():,} → {A.numel()+B.numel():,}  ({100*(A.numel()+B.numel())/W0.numel():.1f}% of original)")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, mat, title in zip(
    axes,
    [W0, A.T, B, dW],
    [f"Frozen W₀  {tuple(W0.shape)}",
     f"A (shown transposed)  {tuple(A.shape)}",
     f"B  {tuple(B.shape)}",
     f"ΔW = B·A  rank={RANK}  {tuple(dW.shape)}"]):
    v = max(mat.abs().max().item(), 1e-8)
    im = ax.imshow(mat.numpy(), cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.05)
    ax.set_title(title, fontsize=9)
plt.suptitle(f"LoRA on '{name.split('.')[-3]}' projection", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Adapter

**Idea:** After the frozen backbone, insert a small trainable bottleneck module:

```
h  →  down(h)  →  GELU  →  up(·)  →  h + residual   →  head
```

- down: ℝᴰ → ℝᵏ  (compress),   up: ℝᵏ → ℝᴰ  (expand),   k ≪ D
- The residual connection means the adapter starts as a near-identity at init.

**Implementation note:** `AdapterHeadClassifier` places the adapter on the pooled
CLS token *after* the full backbone — pedagogically clean. Standard Houlsby adapters
are inserted *inside* each transformer block (after attention and FFN sub-layers).

In [ ]:
K = 32
ad = AdapterHeadClassifier(copy.deepcopy(backbone), D, N_CLASSES, bottleneck_dim=K)

total_p = sum(p.numel() for p in ad.parameters())
train_p = count_trainable_parameters(ad)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.4f}%)")
print(f"Bottleneck: {D} → {K} → {D}  (down: {D*K:,} params,  up: {K*D:,} params)")

W_down = ad.adapter.down.weight.detach().cpu()   # [K, D]
W_up   = ad.adapter.up.weight.detach().cpu()     # [D, K]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, W, title in zip(
    axes,
    [W_down, W_up],
    [f"Down projection  [{K} × {D}]  compress D→k",
     f"Up projection  [{D} × {K}]  expand k→D"]):
    v = W.abs().max().item()
    im = ax.imshow(W.numpy(), cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.02)
    ax.set_title(title)
    ax.set_xlabel("input dim")
    ax.set_ylabel("output dim")
plt.suptitle(f"Adapter bottleneck (k={K}, D={D})", y=1.01)
plt.tight_layout()
plt.show()

# Verify near-identity at init
h = torch.randn(1, D)
with torch.no_grad():
    h_out = ad.adapter(h)
print(f"\nMax residual change at init: {(h_out - h).abs().max().item():.5f}  (near zero by design)")

---
## 6. Partial Fine-tuning

**Idea:** Freeze all layers except the last N transformer blocks and the head.
No new modules — just standard gradient updates on selected pretrained weights.

```
[frozen blocks 0 … L-2]  →  [trainable block L-1]  →  CLS  →  [trainable head]
```

**Tradeoff:** More expressive than LoRA at the same block, but creates a full-size
task-specific checkpoint (no sharing with the base model).

In [ ]:
bb_pf  = copy.deepcopy(backbone)
last_b = list(bb_pf.vit.encoder.layer)[-1]
pft    = PartialFineTuneClassifier(bb_pf, D, N_CLASSES, modules_to_unfreeze=[last_b])

total_p = sum(p.numel() for p in pft.parameters())
train_p = count_trainable_parameters(pft)
print(f"Trainable : {train_p:,} / {total_p:,}  ({100*train_p/total_p:.2f}%)")

# Trainable params per encoder block
block_data = {}
for name, p in pft.backbone.named_parameters():
    if "encoder.layer" not in name:
        continue
    idx = int(name.split("encoder.layer.")[1].split(".")[0])
    d = block_data.setdefault(idx, {"total": 0, "trainable": 0})
    d["total"]     += p.numel()
    d["trainable"] += p.numel() if p.requires_grad else 0

df_pft = pd.DataFrame(block_data).T.sort_index()
df_pft["frozen"] = df_pft["total"] - df_pft["trainable"]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(df_pft.index, df_pft["frozen"],    label="frozen",    color="#abb2bf")
ax.bar(df_pft.index, df_pft["trainable"], label="trainable", color="#e06c75",
       bottom=df_pft["frozen"])
ax.set_xlabel("Transformer block index")
ax.set_ylabel("Parameters")
ax.set_title("Partial fine-tuning: trainable vs frozen parameters per transformer block")
ax.set_xticks(df_pft.index)
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Summary — Parameter Efficiency and Gradient Flow

In [ ]:
_bb0 = copy.deepcopy(backbone)
_bb1 = copy.deepcopy(backbone)
variants = {
    "linear_probe" : LinearProbeModel(copy.deepcopy(backbone), D, N_CLASSES),
    "bitfit"       : BitFitClassifier(copy.deepcopy(backbone), D, N_CLASSES),
    "visual_prompt": PromptTunedClassifier(copy.deepcopy(backbone), D, N_CLASSES, prompt_size=32),
    "lora"         : LoRAClassifier(copy.deepcopy(backbone), D, N_CLASSES,
                                    target_modules=["query", "value"], rank=8),
    "adapter"      : AdapterHeadClassifier(copy.deepcopy(backbone), D, N_CLASSES, bottleneck_dim=32),
    "partial_ft"   : PartialFineTuneClassifier(
                         _bb0, D, N_CLASSES,
                         modules_to_unfreeze=[list(_bb0.vit.encoder.layer)[-1]]),
}

total_p = sum(p.numel() for p in backbone.parameters())
rows = [{"method": n, "trainable": count_trainable_parameters(m),
         "% of backbone": f"{100*count_trainable_parameters(m)/total_p:.3f}%"}
        for n, m in variants.items()]
df_s = pd.DataFrame(rows).set_index("method")
print(df_s.to_string())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(df_s.index, df_s["trainable"], color="steelblue")
ax.axvline(total_p, color="red", linestyle="--", linewidth=1.2,
           label=f"Total backbone ({total_p:,})")
ax.set_xlabel("Trainable parameters")
ax.set_title("Trainable parameter count per method")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def categorise(pname):
    n = pname.lower()
    if "lora_a" in n or "lora_b" in n:          return "LoRA A/B"
    if "adapter" in n:                            return "adapter"
    if "prompt" in n:                             return "prompt"
    if n.startswith("head"):                      return "head"
    if "attention" in n:                          return "backbone attention"
    if "intermediate" in n:                       return "backbone FFN"
    if ".output." in n and "attention" not in n:  return "backbone FFN"
    if "layernorm" in n or "norm" in n:           return "backbone norm"
    if "embedding" in n or "cls_token" in n or "position" in n: return "backbone embed"
    return "other"

def gradient_flow(model, x):
    model.zero_grad()
    y_dummy = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
    F.cross_entropy(model(x), y_dummy).backward()
    buckets = {}
    for pname, param in model.named_parameters():
        cat = categorise(pname)
        has_grad = param.grad is not None and param.grad.abs().sum().item() > 0
        total, live = buckets.get(cat, (0, 0))
        buckets[cat] = (total + param.numel(), live + (param.numel() if has_grad else 0))
    model.zero_grad()
    return {cat: live / max(total, 1) for cat, (total, live) in buckets.items()}

CATS = ["prompt", "LoRA A/B", "adapter",
        "backbone attention", "backbone FFN", "backbone norm", "backbone embed", "head"]

x_probe = torch.randn(2, 3, 224, 224, device=device)
flow = {name: gradient_flow(m, x_probe) for name, m in variants.items()}

grid = pd.DataFrame(
    {name: [flow[name].get(c, 0.0) for c in CATS] for name in variants}, index=CATS)

fig, ax = plt.subplots(figsize=(10, 4))
mat = grid.values.astype(float)
im  = ax.imshow(mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(grid.columns)))
ax.set_xticklabels(grid.columns, rotation=25, ha="right", fontsize=10)
ax.set_yticks(range(len(CATS)))
ax.set_yticklabels(CATS, fontsize=10)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if v > 0:
            lbl = "100%" if v > 0.999 else f"{v*100:.0f}%"
            ax.text(j, i, lbl, ha="center", va="center",
                    fontsize=8, color="black" if v < 0.6 else "white")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02,
             label="fraction of params with non-zero gradient")
ax.set_title("Gradient flow map: which parameter groups receive a gradient?", fontsize=13)
plt.tight_layout()
plt.show()